# Hyperparameter Optimization with Keras (SciKeras)
This notebook demonstrates Grid Search, Random Search, Bayesian optimization with Hyperopt, and optimization with Optuna.

**Demonstration only:** Grid Search and Random Search are configured but not fitted. Hyperopt and Optuna use simulated random losses, not model validation results. Their reported parameters therefore do not identify a better neural network. In a real experiment, train each candidate and evaluate it using the same validation procedure.

All methods use the same three parameters and the same candidate values, defined once below. The number and order of evaluated combinations can differ between methods.


## 1. Install Required Libraries

In [ ]:
#!pip install optuna hyperopt tensorflow scikeras scikit-learn

## 2. Import Necessary Libraries

In [ ]:
import tensorflow as tf
import random
import optuna
from hyperopt import fmin, tpe, hp, Trials, space_eval, STATUS_OK
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scikeras.wrappers import KerasClassifier

# Shared candidate values: change them here for all methods.
PARAM_VALUES = {
    'optimizer': ['adam', 'rmsprop'],
    'epochs': [10, 20, 50, 100, 200, 500, 1000],
    'batch_size': [16, 32, 64]
}


## 3. Define Keras Model and Shared Search Space
The model has one hidden layer with 64 ReLU units and a softmax output layer for 10 classes.

- **Optimizer:** the algorithm that updates the model weights (`adam` or `rmsprop`).
- **Epochs:** the number of complete passes through the training data (10, 20, 50, 100, 200, 500, or 1000).
- **Batch size:** the number of training samples used for one weight update (16, 32, or 64).

There are **2 × 7 × 3 = 42** possible combinations. In SciKeras, `model__optimizer` passes the selected optimizer to the `create_model` function; `epochs` and `batch_size` control training.


In [ ]:
def create_model(optimizer='adam'):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

## 4. Grid Search using SciKeras
Grid Search evaluates every combination of the supplied values. With 42 combinations and 3-fold cross-validation, a real search requires 126 training runs, followed by refitting the best model. It is systematic but can be slow.

The cell below configures the search. Uncomment `fit` and the result lines after preparing `X` and `y`.


In [ ]:
# SciKeras routes the optimizer to the model-building function.
param_grid = {
    'model__optimizer': PARAM_VALUES['optimizer'],
    'epochs': PARAM_VALUES['epochs'],
    'batch_size': PARAM_VALUES['batch_size']
}
model = KerasClassifier(model=create_model, verbose=0)
grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3)
# grid.fit(X, y)
# print('Best Grid Search Parameters:', grid.best_params_)
print('Grid Search candidate values:', param_grid)


## 5. Random Search using SciKeras
Random Search samples a limited number of parameter combinations from the same search space. Here it tests 5 of the 42 combinations using 3-fold cross-validation. It is usually faster than a full grid, but can miss the best combination.

As above, uncomment `fit` and the result line to run actual training.


In [ ]:
# Reuse exactly the same candidate values as Grid Search.
param_dist = param_grid.copy()
random_search = RandomizedSearchCV(
    estimator=model, param_distributions=param_dist,
    n_iter=5, cv=3, random_state=42
)
# random_search.fit(X, y)
# print('Best Random Search Parameters:', random_search.best_params_)
print('Random Search candidate values:', param_dist)


## 6. Bayesian Optimization with Hyperopt
Hyperopt's Tree-structured Parzen Estimator (TPE) uses previous trial results to choose promising parameter combinations. Unlike a fixed grid or random search, later choices depend on earlier results.

Each parameter uses `hp.choice` to select from the shared candidate values. `space_eval` converts Hyperopt's returned choice indices into actual values for **all three parameters**. This demonstration uses simulated losses; replace the objective with model training and validation for meaningful optimization.


In [ ]:
def objective(params):
    # Demonstration only: no model is trained here.
    return {'loss': random.uniform(0.1, 0.5), 'status': STATUS_OK}

space = {
    name: hp.choice(name, values)
    for name, values in PARAM_VALUES.items()
}
trials = Trials()
best_indices = fmin(
    fn=objective, space=space, algo=tpe.suggest,
    max_evals=100, trials=trials
)
best_hyperopt = space_eval(space, best_indices)
print('Hyperopt Parameters (simulated loss):', best_hyperopt)


## 7. Optuna for Keras
Optuna organizes optimization as a sequence of trials. Its default single-objective sampler uses TPE to suggest promising values based on previous results. Optuna also supports pruning to stop unpromising training runs early, although pruning is not used here.

`suggest_categorical` restricts each parameter to exactly the same values used by the other methods. The objective below uses a simulated loss and does not train a Keras model.


In [ ]:
def optuna_objective(trial):
    params = {
        name: trial.suggest_categorical(name, values)
        for name, values in PARAM_VALUES.items()
    }
    # Demonstration only: train and validate a model with params in real use.
    return random.uniform(0.1, 0.5)

study = optuna.create_study(direction='minimize')
study.optimize(optuna_objective, n_trials=100)
print('Optuna Parameters (simulated loss):', study.best_params)
